# Notebook 1: Data Profiling
## Loan Default Risk Analysis | RISK-2024-001
**Author:** Gnanesh (BA)  
**Purpose:** Profile the raw Lending Club dataset — understand shape, missing values, distributions, and potential data quality issues before cleaning.

---
**Read before running:** Follow `data/README.md` to download the dataset first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print('Libraries loaded.')

## 1. Load Data

In [ ]:
# Load a 100,000-row sample for profiling (faster)
# Change nrows=None to load the full dataset
df_raw = pd.read_csv('../data/raw/accepted_loans.csv', low_memory=False, nrows=100000)
print(f'Rows: {len(df_raw):,}')
print(f'Columns: {len(df_raw.columns)}')
df_raw.head(3)

## 2. Dataset Shape and Types

In [ ]:
print('Data Types:')
print(df_raw.dtypes.value_counts())
print('\nMemory Usage:')
print(f"{df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 3. Missing Value Analysis

In [ ]:
# Columns with missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

print(f'Columns with missing values: {len(missing_df)}')
missing_df.head(20)

In [ ]:
# Visualise missing values for analysis columns
ANALYSIS_COLS = [
    'loan_amnt', 'int_rate', 'grade', 'annual_inc', 'dti',
    'emp_length', 'home_ownership', 'purpose', 'term',
    'issue_d', 'loan_status', 'revol_util', 'fico_range_low',
    'verification_status', 'pub_rec_bankruptcies'
]

missing_analysis = missing_pct[ANALYSIS_COLS].sort_values(ascending=False)

ax = missing_analysis.plot(kind='bar', color='steelblue', edgecolor='white')
ax.set_title('Missing Value % — Analysis Columns', fontsize=14, pad=12)
ax.set_ylabel('Missing %')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Target Variable: loan_status

In [ ]:
# Distribution of loan_status
status_counts = df_raw['loan_status'].value_counts()
status_pct = (status_counts / len(df_raw) * 100).round(2)

pd.DataFrame({'count': status_counts, 'pct': status_pct})

In [ ]:
# After filtering to resolved loans only
df_resolved = df_raw[df_raw['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df_resolved['default_flag'] = (df_resolved['loan_status'] == 'Charged Off').astype(int)

print(f'Resolved loans: {len(df_resolved):,}')
print(f'Default rate: {df_resolved["default_flag"].mean()*100:.2f}%')
print(f'Class ratio (non-default:default): {(1-df_resolved["default_flag"].mean())/df_resolved["default_flag"].mean():.1f}:1')

## 5. Numeric Variable Distributions

In [ ]:
# Clean int_rate and revol_util for profiling
df_resolved['int_rate_num'] = df_resolved['int_rate'].astype(str).str.replace('%','').astype(float)

numeric_cols = ['loan_amnt', 'annual_inc', 'dti', 'int_rate_num', 'fico_range_low', 'revol_util']
df_resolved[numeric_cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2)

In [ ]:
# Plot distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    data = df_resolved[col].dropna()
    # Cap extreme outliers for visualisation
    cap = data.quantile(0.99)
    data_capped = data.clip(upper=cap)
    axes[i].hist(data_capped, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col)
    axes[i].set_xlabel('')

plt.suptitle('Distribution of Key Numeric Variables (capped at 99th percentile)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. Categorical Variable Value Counts

In [ ]:
cat_cols = ['grade', 'purpose', 'home_ownership', 'verification_status', 'emp_length', 'term']

for col in cat_cols:
    print(f'\n--- {col} ---')
    print(df_resolved[col].value_counts().to_string())

## 7. Outlier Check

In [ ]:
# Annual income outliers
inc = df_resolved['annual_inc'].dropna()
print(f'Annual income max: {inc.max():,.0f}')
print(f'Annual income 99th pct: {inc.quantile(0.99):,.0f}')
print(f'Rows with income > $500,000: {(inc > 500000).sum()}')

print()

# DTI outliers
dti = df_resolved['dti'].dropna()
print(f'DTI max: {dti.max()}')
print(f'Rows with DTI > 100: {(dti > 100).sum()}')

## 8. Duplicate Check

In [ ]:
dupes = df_resolved.duplicated(subset=['loan_amnt', 'issue_d', 'annual_inc', 'grade']).sum()
print(f'Suspected duplicate rows: {dupes}')

## 9. Summary — Key Data Quality Issues

| Issue | Field | Severity | Handling |
|-------|-------|----------|----------|
| Extreme outliers | annual_inc | High | Cap at 99th percentile |
| Implausible values | dti > 100 | Medium | Cap at 100 |
| Missing values | emp_length ~5.7% | Medium | Treat as 'Unknown' category |
| Missing values | revol_util ~0.5% | Low | Impute with median |
| Missing values | pub_rec_bankruptcies ~0.7% | Low | Treat as 0 |
| Ambiguous target | Current / Late loans | Structural | Exclude from analysis |
| **Selection bias** | **No rejected applications** | **Structural** | **Document as limitation** |

**Next step:** Run `notebooks/02_cleaning.ipynb` to apply all cleaning steps.